# PCA dhe reduktimi i dimensionalitetit

**Lënda:** Machine Learning in Physics  
**Mjedisi:** Python, NumPy, pandas, Matplotlib dhe scikit-learn  
**Roli i notebook-ut:** mjeti kryesor i punës gjatë orës; leksioni i shkruar shërben për thellim teorik.

Në eksperimente moderne mund të masim dhjetëra ose qindra madhësi njëkohësisht, por shumë prej tyre janë të korreluara. PCA pyet: **a mund ta përshkruajmë variacionin kryesor me më pak drejtime?** Kjo është një pyetje lineare-algjebrike, por edhe fizike: cilat kombinime të observablave bartin strukturën dominante dhe sa informacion humbasim kur projektojmë?

Për këtë temë do t'u përgjigjemi sidomos tri pyetjeve:

1. Si lidhet kovarianca me drejtimet kryesore?
2. Çfarë kuptimi ka një eigenvalue në PCA?
3. Kur reduktimi i dimensionalitetit ruan strukturën dhe kur e fsheh atë?

Ritmi i punës është: **parashiko → formulo matematikisht → implemento → ekzekuto → mat → interpreto → provo ta prishësh modelin**. Qelizat nuk duhen ekzekutuar mekanikisht njëra pas tjetrës. Para çdo eksperimenti duhet të kemi një pritshmëri dhe pas tij duhet të shpjegojmë mekanizmin që prodhoi rezultatin.


## Organizimi i seancës

| Koha | Veprimtaria | Rezultati i pritshëm |
|---:|---|---|
| 0–10 min | Pyetja shkencore dhe formulimi matematikor | Kuptimi i objektit që po modelojmë |
| 10–25 min | Zbatimi nga parimet e para | Lidhja ndërmjet formulës dhe algoritmit |
| 25–40 min | Eksperimenti numerik dhe figurat diagnostike | Kontrolli i sjelljes së modelit |
| 40–50 min | Krahasimi me implementimin profesional | Verifikimi i zbatimit dhe praktikës së mirë |
| 50–60 min | Interpretimi, kufizimet dhe një rast dështimi | Përfundim metodologjik, jo vetëm numerik |
| Pas orës | Punishtja dhe ushtrimet | Ndrysho një parametër, mat efektin dhe shpjego mekanizmin |

Kjo tabelë është orientuese. Nëse një diskutim fizik ose matematik është i frytshëm, është më e vlefshme të ndalemi te ai sesa të “mbarojmë” notebook-un.


## Rezultatet e të nxënit

Në fund të punës studenti duhet të jetë në gjendje të:

- Ndërtoj matricën e kovariancës.
- Nxjerr eigenvalues/eigenvectors.
- Interpretoj explained variance.
- Projektoj dhe rikonstruktoj të dhënat.
- Dalloj kompresimin matematikor nga interpretimi fizik.


### Para se të ekzekutojmë: Mjedisi llogaritës dhe riprodhueshmëria

Ngarkojmë bibliotekat që do të përdoren dhe fiksojmë `RANDOM_STATE = 42`. Ky seed nuk e bën problemin determinist në kuptimin fizik; ai thjesht na lejon të përsërisim të njëjtin eksperiment llogaritës dhe të diskutojmë metodën pa u hutuar nga ndryshime të rastësishme të mostrës.

**Parashikim para ekzekutimit.** Vlerësoni rendin e madhësisë dhe drejtimin e rezultatit përpara se ta llogarisë kompjuteri.


In [ ]:
# Qëllimi i këtij blloku: Mjedisi llogaritës dhe riprodhueshmëria.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import numpy as np
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import pandas as pd
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import matplotlib.pyplot as plt
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from pathlib import Path
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("08 — PCA dhe reduktimi i dimensionalitetit")
print("Seed:", RANDOM_STATE)

**Interpretim.** Një dalje numerike merr kuptim vetëm pasi krahasohet me shkallën e problemit, pritshmërinë teorike dhe një kontroll të pavarur kur është e mundur.


# 1. Spektra të korreluara — CORE

### Para se të ekzekutojmë: Spektra me pak faktorë latentë

Ndërtojmë spektra me shumë gjatësi valore, por me vetëm dy amplituda kryesore që ndryshojnë. Pra dimensioni i matur është i madh, ndërsa dimensioni efektiv i strukturës është i vogël. Ky është skenari ideal për të kuptuar PCA.

**Parashikim para ekzekutimit.** Vendosni çfarë forme duhet të ketë figura nëse argumenti teorik është i saktë. Grafiku përdoret për të testuar një pritshmëri, jo vetëm për ta ilustruar.


In [ ]:
# Qëllimi i këtij blloku: Spektra me pak faktorë latentë.
wl=np.linspace(400,750,120)
n=260
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
a=rng.normal(1.0,0.12,n)
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
b=rng.normal(0.5,0.09,n)
spectra=[]
for ai,bi in zip(a,b):
    s=ai*np.exp(-0.5*((wl-520)/28)**2)+bi*np.exp(-0.5*((wl-640)/35)**2)
    # Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
    s+=rng.normal(0,0.02,len(wl))
    spectra.append(s)
X=np.array(spectra)
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
# Përsërisim procedurën për të studiuar konvergjencën ose ndjeshmërinë ndaj parametrave.
for i in range(20):
    ax.plot(wl,X[i],alpha=0.5)
ax.set_xlabel("Gjatësia valore [nm]"); ax.set_ylabel("Intensitet")
ax.set_title("Shembuj spektrash")
plt.show()

**Interpretim.** Figura duhet lexuar si test i një hipoteze të vogël: a përputhet forma e vëzhguar me atë që parashikuam nga matematika?


# 2. Nga kovarianca te drejtimet kryesore — CORE

Pas qendrimit $X_c=X-\bar X$, matrica e kovariancës është
$$
C=\frac{1}{N-1}X_c^TX_c.
$$
Drejtimet kryesore plotësojnë
$$
C\mathbf v_i=\lambda_i\mathbf v_i.
$$
Vlera vetjake $\lambda_i$ mat variancën përgjatë komponentit përkatës.

### Para se të ekzekutojmë: Ndërtimi manual i PCA

Qendrojmë të dhënat, ndërtojmë kovariancën dhe diagonalizojmë matricën. Renditja e eigenvalues nga më e madhja te më e vogla identifikon drejtimet e variacionit maksimal.

**Parashikim para ekzekutimit.** Nëse të dhënat janë të forta të korreluara, prisni që pak eigenvalues të mbajnë pjesën më të madhe të variancës.


In [ ]:
# Qëllimi i këtij blloku: Ndërtimi manual i PCA.
Xc=X-X.mean(axis=0)
# Kovarianca përmbledh bashkë-ndryshimin e veçorive pas qendrimit të të dhënave.
C=np.cov(Xc,rowvar=False)
# Diagonalizojmë matricën simetrike të kovariancës për të marrë komponentët kryesorë.
vals,vecs=np.linalg.eigh(C)
# Renditja përdoret për të identifikuar fqinjët ose komponentët sipas kriterit përkatës.
idx=np.argsort(vals)[::-1]
vals,vecs=vals[idx],vecs[:,idx]
explained=vals/vals.sum()

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.semilogy(np.arange(1,21),vals[:20],marker="o")
ax.set_xlabel("Komponenti"); ax.set_ylabel("Vlera vetjake")
ax.set_title("Spektri i vlerave vetjake")
plt.show()

**Interpretim.** Komponentët me eigenvalues të mëdha përshkruajnë drejtimet dominante të variacionit. Kjo nuk do të thotë automatikisht se ata janë mekanizma fizikë të pavarur.


### PCA nga paketa didaktike `src/`

### Para se të ekzekutojmë: Krahasim me modulin kanonik

Përdorim `PCAFromScratch` për të verifikuar se procedura e paketizuar jep të njëjtën variancë të shpjeguar. Kjo është një kontroll i drejtpërdrejtë i implementimit manual.

**Parashikim para ekzekutimit.** Nëse të dhënat janë të forta të korreluara, prisni që pak eigenvalues të mbajnë pjesën më të madhe të variancës.


In [ ]:
# Qëllimi i këtij blloku: Krahasim me modulin kanonik.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import sys
# Kërkojmë automatikisht rrënjën e repository-t, në mënyrë që notebook-u të funksionojë nga dosje të ndryshme.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from src.pca.pca_from_scratch import PCAFromScratch
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
pca_src = PCAFromScratch(n_components=2).fit(X)
print("Varianca e shpjeguar (src):", np.round(pca_src.explained_variance_ratio_, 5))

**Interpretim.** Komponentët me eigenvalues të mëdha përshkruajnë drejtimet dominante të variacionit. Kjo nuk do të thotë automatikisht se ata janë mekanizma fizikë të pavarur.


# 3. Varianca kumulative — CORE

### Para se të ekzekutojmë: Sa komponentë ruajmë?

Varianca kumulative e shpjeguar është një mënyrë praktike për të vendosur sa dimensione të ruajmë. Pragu 90% ose 95% është konvencion, jo ligj fizik; zgjedhja duhet lidhur me qëllimin e analizës.

**Parashikim para ekzekutimit.** Vendosni çfarë forme duhet të ketë figura nëse argumenti teorik është i saktë. Grafiku përdoret për të testuar një pritshmëri, jo vetëm për ta ilustruar.


In [ ]:
# Qëllimi i këtij blloku: Sa komponentë ruajmë?.
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(np.cumsum(explained[:20]),marker="o")
ax.set_xlabel("Numri i komponentëve")
ax.set_ylabel("Varianca kumulative e shpjeguar")
ax.set_ylim(0,1.02)
ax.set_title("Sa dimensione duhen?")
plt.show()

**Interpretim.** Figura duhet lexuar si test i një hipoteze të vogël: a përputhet forma e vëzhguar me atë që parashikuam nga matematika?


# 4. Projekcioni në hapësirën PCA — CORE

### Para se të ekzekutojmë: Projekcioni në koordinata të reja

Koordinatat PCA janë projektime të spektrave mbi drejtimet kryesore. Ngjyrosja me një parametër latent na ndihmon të shohim nëse komponenti ka lidhje me një variacion fizik të njohur.

**Parashikim para ekzekutimit.** Vendosni çfarë forme duhet të ketë figura nëse argumenti teorik është i saktë. Grafiku përdoret për të testuar një pritshmëri, jo vetëm për ta ilustruar.


In [ ]:
# Qëllimi i këtij blloku: Projekcioni në koordinata të reja.
Z=Xc@vecs[:,:2]
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
scat=ax.scatter(Z[:,0],Z[:,1],c=a,s=16)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Projekcioni PCA; ngjyra kodon amplitudën e pikut të parë")
plt.colorbar(scat,ax=ax,label="a")
plt.show()

**Interpretim.** Figura duhet lexuar si test i një hipoteze të vogël: a përputhet forma e vëzhguar me atë që parashikuam nga matematika?


# 5. Loadings dhe kuptimi fizik — CORE

### Para se të ekzekutojmë: Loadings dhe interpretimi spektral

Loadings tregojnë cilat gjatësi valore kontribuojnë te një komponent. Struktura e tyre mund të ketë kuptim fizik, por komponenti është përcaktuar nga varianca, jo nga një model shkakësor.

**Parashikim para ekzekutimit.** Vendosni çfarë forme duhet të ketë figura nëse argumenti teorik është i saktë. Grafiku përdoret për të testuar një pritshmëri, jo vetëm për ta ilustruar.


In [ ]:
# Qëllimi i këtij blloku: Loadings dhe interpretimi spektral.
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(wl,vecs[:,0],label="PC1 loading")
ax.plot(wl,vecs[:,1],label="PC2 loading")
ax.set_xlabel("Gjatësia valore [nm]")
ax.set_ylabel("Koeficienti")
ax.legend(); ax.set_title("Çfarë strukturash spektrale kap PCA?")
plt.show()

**Interpretim.** Figura duhet lexuar si test i një hipoteze të vogël: a përputhet forma e vëzhguar me atë që parashikuam nga matematika?


# 6. Rikonstruksioni — CORE

### Para se të ekzekutojmë: Rikonstruksioni dhe humbja e informacionit

Rikonstruktojmë të njëjtin spektër me numër në rritje komponentësh. Kjo e bën të dukshme çfarë informacioni ruhet dhe çfarë hidhet poshtë gjatë reduktimit dimensional.

**Parashikim para ekzekutimit.** Vendosni çfarë forme duhet të ketë figura nëse argumenti teorik është i saktë. Grafiku përdoret për të testuar një pritshmëri, jo vetëm për ta ilustruar.


In [ ]:
# Qëllimi i këtij blloku: Rikonstruksioni dhe humbja e informacionit.
sample=7
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(wl,X[sample],label="origjinali")
for k in [1,2,5,15]:
    recon=X.mean(axis=0)+(Xc[sample]@vecs[:,:k])@vecs[:,:k].T
    ax.plot(wl,recon,label=f"k={k}")
ax.legend(); ax.set_xlabel("nm"); ax.set_ylabel("Intensitet")
ax.set_title("Rikonstruksioni me numër të ndryshëm komponentësh")
plt.show()

**Interpretim.** Figura duhet lexuar si test i një hipoteze të vogël: a përputhet forma e vëzhguar me atë që parashikuam nga matematika?


---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 7. Krahasim me sklearn — EXTENDED

### Para se të ekzekutojmë: Krahasimi me implementimin profesional

Në fund verifikojmë raportet e variancës me `sklearn`. Shenja e eigenvectors mund të ndryshojë, por nën-hapësira dhe eigenvalues duhet të përputhen.

**Parashikim para ekzekutimit.** Nëse të dhënat janë të forta të korreluara, prisni që pak eigenvalues të mbajnë pjesën më të madhe të variancës.


In [ ]:
# Qëllimi i këtij blloku: Krahasimi me implementimin profesional.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.decomposition import PCA
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
p=PCA(n_components=5).fit(X)
print("Manual:",np.round(explained[:5],5))
print("sklearn:",np.round(p.explained_variance_ratio_,5))

**Interpretim.** Komponentët me eigenvalues të mëdha përshkruajnë drejtimet dominante të variacionit. Kjo nuk do të thotë automatikisht se ata janë mekanizma fizikë të pavarur.


# 8. Aplikim jo-fizik: të dhëna biometrike — EXTENDED

Në biologji ose mjekësi, qindra matje të korreluara mund të reduktohen në pak drejtime dominante. Kjo lehtëson vizualizimin, por komponentët nuk janë automatikisht variabla kauzale.

## Punishtja — ndrysho, mat dhe shpjego

Në këtë pjesë nuk kërkohet ekzekutim pasiv. Për çdo eksperiment:

1. shkruani paraprakisht çfarë prisni të ndodhë;
2. ndryshoni **vetëm një** element në të njëjtën kohë;
3. ruani metrikën ose figurën që ndryshon;
4. shpjegoni mekanizmin matematik ose statistikor;
5. dalloni qartë atë që tregon eksperimenti nga ajo që **nuk** mund të përfundohet prej tij.

### Pyetje për diskutim

1. Pse komponentët kryesorë janë ortogonalë?
2. A janë PCA loadings domosdoshmërisht madhësi fizike?
3. Pse standardizimi para PCA mund të ndryshojë përgjigjen?
4. Si lidhet explained variance me gabimin e rikonstruksionit?


# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Verifikoni ortogonalitetin e komponentëve të parë.
2. Llogaritni gabimin e rikonstruksionit ndaj k.
3. Standardizoni çdo gjatësi valore dhe përsëritni PCA.
4. Shpjegoni pse shenja e një eigenvector-i është arbitrare.

        ## B. Ushtrim i plotë
        Shtoni një pik spektral të tretë me amplitudë të pavarur dhe kontrolloni sa komponentë duhen për 95% variancë.

        ## C. Playground
        Ndryshoni numrin e mostrave, zhurmën, amplitudat dhe numrin e komponentëve.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.

## Përmbledhje konceptuale e kapitullit

Notebook-u **PCA dhe reduktimi i dimensionalitetit** duhet të lihet me një ide të qartë: algoritmi është vetëm një hallkë e arsyetimit. Para tij kemi zgjedhjen e përfaqësimit dhe hipotezat mbi të dhënat; pas tij kemi validimin, diagnostikën dhe interpretimin. Nëse një student mund të ekzekutojë të gjitha qelizat por nuk mund të shpjegojë pse secila prej tyre është e nevojshme, atëherë puna llogaritëse nuk është përfunduar ende.

Për studim të mëtejshëm, rikthehuni te shënimet e leksionit për derivimet e plota dhe përdoreni këtë notebook si laborator ku këto ide testohen numerikisht.

## Kontrolli përfundimtar

- [ ] Ndërtoj matricën e kovariancës.
- [ ] Nxjerr eigenvalues/eigenvectors.
- [ ] Interpretoj explained variance.
- [ ] Projektoj dhe rikonstruktoj të dhënat.
- [ ] Dalloj kompresimin matematikor nga interpretimi fizik.
